# STEP 2 — Axis 1: Accuracy (Direct vs CoT)

**What this step does:**
- Runs all 7 models on GSM8K + BBH + TruthfulQA with two prompting modes: `direct` and `cot`
- Each model is loaded, evaluated, and then unloaded before loading the next (T4 memory management)

**Outputs produced:**
- `results/accuracy_raw.csv` — per-question predictions
- `results/accuracy_summary.csv` — accuracy per (model, mode, dataset)

**Estimated run time:** ~4–6 hours (all 7 models × 3 datasets × 2 modes)

**Prerequisite:** Step 1 must have completed (`data/datasets.json` must exist)

In [ ]:
import sys, json, gc
import torch
import pandas as pd
from pathlib import Path

sys.path.insert(0, '/kaggle/working/light_paper')

BASE_DIR    = Path('/kaggle/working/light_paper')
DATA_DIR    = BASE_DIR / 'data'
RESULTS_DIR = BASE_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

from src.models    import MODEL_REGISTRY, load_model
from src.prompts   import make_accuracy_prompt
from src.inference import generate_responses, parse_gsm8k_answer, parse_bbh_answer, parse_truthfulqa_answer
from src.metrics   import compute_accuracy_by_dataset

# Load datasets
with open(DATA_DIR / 'datasets.json') as f:
    all_data = json.load(f)

MODES = ['direct', 'cot']
PARSERS = {
    'gsm8k':     parse_gsm8k_answer,
    'bbh':       parse_bbh_answer,
    'truthfulqa': parse_truthfulqa_answer,
}

print('Datasets loaded:')
for k, v in all_data.items():
    print(f'  {k}: {len(v)} samples')

In [ ]:
# ── Resume-safe: load existing rows if re-running after interruption ──────
RAW_PATH = RESULTS_DIR / 'accuracy_raw.csv'

if RAW_PATH.exists():
    existing_df = pd.read_csv(RAW_PATH)
    done_keys = set(zip(existing_df['model'], existing_df['mode'], existing_df['dataset']))
    print(f'⚡ Resuming — {len(done_keys)} (model, mode, dataset) combos already done')
else:
    existing_df = pd.DataFrame()
    done_keys = set()

all_rows = existing_df.to_dict('records') if not existing_df.empty else []

In [ ]:
# ── Main evaluation loop ──────────────────────────────────────────────────
for model_config in MODEL_REGISTRY:
    for mode in MODES:
        # Check what's not yet done for this model+mode
        pending_datasets = [
            ds for ds in ['gsm8k', 'bbh', 'truthfulqa']
            if (model_config.name, mode, ds) not in done_keys
        ]
        if not pending_datasets:
            print(f'✓ Skip {model_config.name}/{mode} (all done)')
            continue

        print(f'\n── {model_config.name} | mode={mode} ─────────────────────────')
        model, tokenizer = load_model(model_config)

        for ds_name in pending_datasets:
            samples  = all_data[ds_name]
            parser   = PARSERS[ds_name]

            prompts  = [make_accuracy_prompt(s['question'], mode, ds_name) for s in samples]
            print(f'  [{ds_name}] {len(prompts)} prompts...')

            responses = generate_responses(
                model, tokenizer, prompts,
                max_new_tokens=512, batch_size=4
            )

            for s, resp in zip(samples, responses):
                predicted = parser(resp)
                all_rows.append({
                    'model':     model_config.name,
                    'size_b':    model_config.size_b,
                    'mode':      mode,
                    'dataset':   ds_name,
                    'id':        s['id'],
                    'question':  s['question'][:200],
                    'answer':    s['answer'],
                    'predicted': predicted if predicted else '',
                    'raw_response': resp[:500],
                })

            # Save after each dataset — safe against interruption
            df = pd.DataFrame(all_rows)
            df.to_csv(RAW_PATH, index=False)
            done_keys.add((model_config.name, mode, ds_name))

            # Print current accuracy
            subset = df[(df.model == model_config.name) & (df.mode == mode) & (df.dataset == ds_name)]
            acc = (subset['predicted'].str.lower().str.strip() == subset['answer'].str.lower().str.strip()).mean()
            print(f'  [{ds_name}] acc = {acc:.3f}')

        # Unload model before loading next
        del model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()
        print(f'  GPU freed')

In [ ]:
# ── Compute and save accuracy summary ────────────────────────────────────
df = pd.read_csv(RAW_PATH)

# Add correct column
df['correct'] = (
    df['predicted'].fillna('').str.lower().str.strip() ==
    df['answer'].fillna('').str.lower().str.strip()
)

summary = compute_accuracy_by_dataset(df)
summary_path = RESULTS_DIR / 'accuracy_summary.csv'
summary.to_csv(summary_path, index=False)

print(summary.pivot_table(index=['model','size_b'], columns=['mode','dataset'], values='accuracy').round(3).to_string())
print(f'\n✅ STEP 2 COMPLETE')
print(f'   Outputs: results/accuracy_raw.csv, results/accuracy_summary.csv')
print(f'   Next: Run step3_calibration.ipynb')